# Chapter 02 — Advanced Transforms, Side Inputs & BigQuery I/O

Welcome to the interactive notebook for **Chapter 02**!

> 📖 **Concept Reference**: For complete explanations of the `DoFn` lifecycle, Side Inputs broadcasting, and Dead-Letter Queue patterns, see [Chapter 02 README](README.md#core-concepts-deep-dive).

### Interactive Workflow
1. Inspect raw sales transactions and the auxiliary product catalog.
2. Parse the product catalog into an in-memory Side Input (`beam.pvalue.AsDict`).
3. Implement validation and branch poison records using `beam.pvalue.TaggedOutput`.
4. Stream curated records into **Google BigQuery** using the Storage Write API.

## 1. Environment Setup & Kernel Selection

This project uses **[`uv`](https://docs.astral.sh/uv/)** to manage dependencies in an isolated `.venv` virtual environment.

> 📘 **Full Environment Setup Guide**: For complete setup instructions, terminal commands, and kernel registration, see the root [README.md](../README.md#environment-setup--activating-the-kernel-with-uv).

* **Option A: In the Terminal**: Run `source .venv/bin/activate` or execute commands directly with `uv run`.
* **Option B: In This Notebook (Kernel Selection)**: Click the **Kernel Picker** in the top-right corner and select **`Python (dataflow-workshops)`** (or browse to `.venv/bin/python`).

Run the verification cell below to confirm your setup:


In [ ]:
import os
import sys

# Verify Python executable location
print(f"Python Executable: {sys.executable}")
if ".venv" in sys.executable:
    print("✅ Success: Notebook is running inside the project's .venv!")
else:
    print("⚠️ Warning: Notebook is running on system Python! Select 'Python (dataflow-workshops)' from the top-right kernel picker.")

import apache_beam as beam
print(f"✅ Apache Beam Version: {beam.__version__}")

## 2. Inspecting the Raw Data & Reference Dimensions

In this chapter, we work with two datasets located in `data/`:
1. **`data/sales_data_with_errors.csv`**: Raw retail transactions containing valid records and intentional corruptions (negative amounts, missing product IDs, malformed dates).
2. **`data/product_catalog.json`**: A product dimension catalog containing metadata (`product_name`, `category`, `unit_cost`).

Let's inspect both files:

In [ ]:
from pathlib import Path

print("=== [1] Raw Sales Transactions (with errors) ===")
print(Path("data/sales_data_with_errors.csv").read_text())

print("=== [2] Product Dimension Catalog ===")
print(Path("data/product_catalog.json").read_text())

## 3. Implementing the `DoFn` Lifecycle

We structure worker initialization with `setup()` and clean teardown with `teardown()`.
*(For the complete lifecycle hook reference table, see [Chapter 02 README](README.md#1-dofn-lifecycle-methods)).*

## 4. Side Inputs (`beam.pvalue.AsDict`)

We broadcast the dimension catalog to all workers as an in-memory dictionary, achieving $O(1)$ lookups without expensive join shuffles.
*(For architectural details on Side Inputs vs. CoGroupByKey, see [Chapter 02 README](README.md#2-side-inputs-beampvalueasdict)).*

In [ ]:
import json

class ParseCatalogFn(beam.DoFn):
    """Parses JSON catalog lines into (product_id, metadata) pairs for Side Input."""
    def process(self, line: str):
        line = line.strip()
        if not line or line in ("[", "]"):
            return
        clean_line = line.rstrip(",")
        try:
            item = json.loads(clean_line)
            if "product_id" in item:
                yield (item["product_id"], item)
        except json.JSONDecodeError:
            pass

## 5. Branching with Tagged Multi-Outputs & Dead-Letter Queues (DLQ)

We route clean records to the main stream and quarantine poison records to `dead_letter` using `beam.pvalue.TaggedOutput`.
*(For error routing patterns and schema isolation, see [Chapter 02 README](README.md#3-tagged-multi-outputs--dead-letter-queues-dlq)).*

In [ ]:
import datetime
import re

DEAD_LETTER_TAG = "dead_letter"

class ValidateAndEnrichFn(beam.DoFn):
    def setup(self):
        # Compiled once per worker instance
        self.date_regex = re.compile(r"^\d{4}-\d{2}-\d{2}$")
        
    def process(self, line: str, catalog: dict):
        line = line.strip()
        if not line or line.startswith("transaction_id"):
            return
            
        parts = [p.strip() for p in line.split(",")]
        now_ts = datetime.datetime.now(datetime.timezone.utc).isoformat()
        
        # Rule 1: Field count
        if len(parts) != 4:
            yield beam.pvalue.TaggedOutput(DEAD_LETTER_TAG, {
                "raw_line": line,
                "error_reason": f"Expected 4 fields, got {len(parts)}",
                "timestamp": now_ts
            })
            return
            
        tx_id, prod_id, amount_str, tx_date = parts
        
        # Rule 2: Missing product ID
        if not prod_id:
            yield beam.pvalue.TaggedOutput(DEAD_LETTER_TAG, {
                "raw_line": line,
                "error_reason": "Missing product_id",
                "timestamp": now_ts
            })
            return
            
        # Rule 3: Valid positive amount
        try:
            amount = float(amount_str)
            if amount <= 0:
                yield beam.pvalue.TaggedOutput(DEAD_LETTER_TAG, {
                    "raw_line": line,
                    "error_reason": f"Non-positive amount: {amount}",
                    "timestamp": now_ts
                })
                return
        except ValueError:
            yield beam.pvalue.TaggedOutput(DEAD_LETTER_TAG, {
                "raw_line": line,
                "error_reason": f"Invalid numeric amount '{amount_str}'",
                "timestamp": now_ts
            })
            return
            
        # Rule 4: Valid date format
        if not self.date_regex.match(tx_date):
            yield beam.pvalue.TaggedOutput(DEAD_LETTER_TAG, {
                "raw_line": line,
                "error_reason": f"Invalid date format '{tx_date}' (must be YYYY-MM-DD)",
                "timestamp": now_ts
            })
            return
            
        # Rule 5: Side Input lookup in catalog
        prod_info = catalog.get(prod_id)
        if not prod_info:
            yield beam.pvalue.TaggedOutput(DEAD_LETTER_TAG, {
                "raw_line": line,
                "error_reason": f"Product '{prod_id}' not found in catalog",
                "timestamp": now_ts
            })
            return
            
        # Successful enrichment emitted to default (main) stream
        unit_cost = prod_info.get("unit_cost", 0.0)
        profit = round(amount - unit_cost, 2)
        
        yield {
            "transaction_id": tx_id,
            "product_id": prod_id,
            "product_name": prod_info.get("product_name"),
            "category": prod_info.get("category"),
            "amount": amount,
            "profit": profit,
            "transaction_date": tx_date,
            "processed_at": now_ts,
        }

## 6. Pipeline Architecture (DAG)

Notice how the **`PCollection`** (cylinders) and **`PTransform`** (rectangles) branch into two destinations:

![Chapter 02 Pipeline Architecture](../artifacts/02_side_inputs_and_bq_dag.png)

<details>
<summary><b>Click to expand raw Mermaid flowchart definition</b></summary>

```mermaid
flowchart LR
    classDef pcoll fill:#e8f0fe,stroke:#4285f4,stroke-width:2px,color:#1a73e8,font-weight:bold;
    classDef ptrans fill:#fef7e0,stroke:#f9ab00,stroke-width:2px,color:#b06000,font-weight:bold;
    classDef sink fill:#e6f4ea,stroke:#34a853,stroke-width:2px,color:#137333,font-weight:bold;
    classDef dlq fill:#fce8e6,stroke:#ea4335,stroke-width:2px,color:#c5221f,font-weight:bold;

    subgraph Ingest["1. Inputs & Side Input"]
        SALES[("sales_data_with_errors.csv")]:::pcoll
        CATALOG[("product_catalog.json")]:::pcoll
        T_CAT["'ReadCatalog'<br/><i>ParseCatalogFn</i>"]:::ptrans
        PC_CAT[("Catalog Map")]:::pcoll
        CATALOG --> T_CAT --> PC_CAT
    end

    subgraph Processing["2. Validate, Enrich & Branch"]
        T_SALES["'ReadSales'<br/><i>ReadFromText</i>"]:::ptrans
        T_ENRICH["'ValidateAndEnrich'<br/><i>ValidateAndEnrichFn</i>"]:::ptrans
        PC_VALID[("Valid Enriched Sales")]:::pcoll
        PC_DLQ[("Poison Records DLQ")]:::dlq
        SALES --> T_SALES --> T_ENRICH
        PC_CAT -. "Side Input: AsDict" .-> T_ENRICH
        T_ENRICH -- "Main Output" --> PC_VALID
        T_ENRICH -- "Tagged Output ('dead_letter')" --> PC_DLQ
    end

    subgraph Sinks["3. Dual-Sink Persistence"]
        T_BQ["'WriteToBigQuery'<br/><i>Storage Write API</i>"]:::ptrans
        BQ[("BigQuery: curated_sales")]:::sink
        T_DLQ["'WriteDLQ'<br/><i>WriteToText</i>"]:::ptrans
        GCS_DLQ[("GCS: dead_letter-*.json")]:::dlq
        PC_VALID --> T_BQ --> BQ
        PC_DLQ --> T_DLQ --> GCS_DLQ
    end
```
</details>

## 7. Executing the Pipeline Locally with `DirectRunner`

Let's run the complete pipeline locally and write outputs to `output/`:

In [ ]:
import shutil

output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

valid_prefix = "output/notebook_valid_sales"
dlq_prefix = "output/notebook_dlq_records"

with beam.Pipeline() as p:
    # 1. Build Side Input Collection
    catalog_pcoll = (
        p
        | "ReadCatalog" >> beam.io.ReadFromText("data/product_catalog.json")
        | "ParseCatalog" >> beam.ParDo(ParseCatalogFn())
    )
    
    # 2. Validate, Enrich, and Branch
    results = (
        p
        | "ReadSales" >> beam.io.ReadFromText("data/sales_data_with_errors.csv")
        | "ValidateAndEnrich" >> beam.ParDo(
            ValidateAndEnrichFn(),
            catalog=beam.pvalue.AsDict(catalog_pcoll)
        ).with_outputs(DEAD_LETTER_TAG, main="valid_sales")
    )
    
    valid_sales = results.valid_sales
    dead_letter = results[DEAD_LETTER_TAG]
    
    # 3. Write Valid Records Locally
    (
        valid_sales
        | "FormatValid" >> beam.Map(json.dumps)
        | "WriteValid" >> beam.io.WriteToText(valid_prefix, file_name_suffix=".json")
    )
    
    # 4. Write Poison Records to Dead-Letter Queue Locally
    (
        dead_letter
        | "FormatDLQ" >> beam.Map(json.dumps)
        | "WriteDLQ" >> beam.io.WriteToText(dlq_prefix, file_name_suffix=".json")
    )

print("✅ Pipeline completed locally!")

Let's inspect the results:

In [ ]:
print("=== [1] Valid Enriched Sales Records ===")
for f in Path("output").glob("notebook_valid_sales*.json"):
    print(f.read_text())

print("=== [2] Dead-Letter Poison Records ===")
for f in Path("output").glob("notebook_dlq_records*.json"):
    print(f.read_text())

## 8. Ingesting into Google Cloud BigQuery

We persist curated sales records into BigQuery using the high-throughput `STORAGE_WRITE_API` method.
*(For schema mapping and quota optimization, see [Chapter 02 README](README.md#4-bigquery-storage-write-api)).*

## Summary & What's Next

In this chapter, you mastered:
* **`DoFn` lifecycle methods** (`setup`, `process`, `teardown`) to properly manage worker resources.
* **Side Inputs (`beam.pvalue.AsDict`)** to perform $O(1)$ dimension table lookups without expensive distributed shuffles.
* **Tagged Multi-Outputs (`beam.pvalue.TaggedOutput`)** to isolate corrupt records into a Dead-Letter Queue without crashing the pipeline.
* **BigQuery Storage Write API** for scalable, cost-effective data warehousing.

👉 **Next: [Chapter 03 — Real-Time Streaming & Windowing](../03_realtime_streaming/README.md)**:
We transition from batch files to continuous unbounded streaming from **Google Cloud Pub/Sub**, exploring **Fixed/Sliding Windows**, **Watermarks**, and **Late-Data Handling**!